In [ ]:
import sys
sys.path.join("../")

In [98]:
import geemap
import ee
import pandas as pd
import numpy as np
import xarray as xr
from ipyleaflet.velocity import Velocity
import hvplot.xarray
from datetime import datetime
from tqdm import tqdm
import gdown
from tools import get_band_info
import calendar

In [99]:
try:
    ee.Initialize()
except:
    ee.Authenticate()
    ee.Initialize()

In [12]:
point = ee.Geometry.Point([-51.51, -29.93])  # Longitude, Latitude
hydrobasins = ee.FeatureCollection('WWF/HydroATLAS/v1/Basins/level05')
basin = hydrobasins.filterBounds(point)

In [5]:
# dataset = ee.ImageCollection("NASA/GPM_L3/IMERG_V06").select("precipitationCal")

# year = 2010
# month = 1

# ds = []

# tbounds = pd.date_range(start="2000-06-01",end="2024-07-26", freq="90d")
# for ti, tj in tqdm(zip(tbounds[:-1], tbounds[1:]), total=tbounds.size-1):
#     df = geemap.ee_to_df(geemap.zonal_stats(dataset.filterDate(ti,tj), basin, stat_type="MEAN", return_fc=True, verbose=False))
    
#     # Extract time from column names and create a datetime column
#     time_pattern = r"(\d{12})"  # Pattern to match YYYYMMDD
#     datetime =  df.columns.str.extract(time_pattern, expand=False)
#     datetime = pd.to_datetime(datetime, format='%Y%m%d%H%M')
    
#     valid = ~np.isnan(datetime)
#     varinfo = get_band_info(info[name]["id"]).loc[info[name]["variable"]][["Units","Description"]].to_dict()
#     varinfo = {key.lower():varinfo[key] for key in varinfo}
#     dsi = xr.DataArray(df.values[0][valid].astype("float"), dims=["time"] ,coords=dict(time=datetime[valid])).rename("precipitation")
#     dsi.attrs = varinfo
#     ds.append(dsi)
# ds = xr.concat(ds, "time")

In [362]:
# from scipy.signal import find_peaks
# import holoviews as hv
# peaks, props = find_peaks(ds.values, width=5)

In [6]:
# ds.groupby("time.year").mean().plot()

In [452]:
# threshold = 0.1

# val = ds.values

# ddt = np.diff((val>0.1).astype("int"))
# left = np.argwhere(ddt>0).ravel()
# right = np.argwhere(ddt<0).ravel()


# peaks = []
# vmax = []
# duration = []
# total = []
# dt = 0.5
# for lefti, righti in zip(left, right):
#     vali = val[lefti:righti+1]
#     vmax.append(vali.max())
#     peaks.append(np.argmax(vali)+lefti)
#     duration.append(len(vali)*dt)
#     total.append(vali.sum())
# vmax = np.array(vmax)
# peaks = np.array(peaks)
# duration = np.array(duration)
# total = np.array(total)

In [453]:
# vmin = 1
# where = vmax>vmin

# vmax = vmax[where]
# peaks = peaks[where]
# left = left[where]
# right = right[where]
# duration = duration[where]
# total = total[where]

In [7]:
# plt.hist(total, bins=np.arange(0,200,2));

In [100]:
percentiles = np.arange(0,100+1).astype("int").tolist()

datasets_all = {
    'NASA/GSFC/MERRA/slv/2': ["PS", "T10M", "TQV", "U10M", "V10M", "H500", "U500", "V500"],
}

In [102]:
lonlim = [-180, 180]
latlim = [-80, 12+58/60]
bbox = ee.Geometry.BBox(lonlim[0], latlim[0], lonlim[1], latlim[1])

In [103]:
for dataset_id in datasets_all:
    variables = datasets_all[dataset_id]
    dataset_name = dataset_id.replace("/","_")

    datasets = {
        f"percentiles_{dataset_name}_all_extended": (
            ee.ImageCollection(dataset_id).select(variables)
                .reduce(ee.Reducer.percentile(percentiles))
        ),
        f"percentiles_{dataset_name}_apr_may_extended": (
            ee.ImageCollection(dataset_id).select(variables)
                .filter(ee.Filter.dayOfYear(91, 151))
                .reduce(ee.Reducer.percentile(percentiles))
        ),
    }
 
    for dataset in datasets:
        
        print(dataset)
        
        image = datasets[dataset]
        
        # Export the image, specifying the CRS, transform, and region.
        task = ee.batch.Export.image.toDrive(
            image=image,
            description=dataset,
            crs="EPSG:4326",
            folder="export",
            region=bbox,
        )
        task.start()


path = f"../data/external/gpm_merra2"
dataset_name = dataset_id.replace("/","_")
dataset = ee.ImageCollection(dataset_id).select(variables).filter(ee.Filter.date("2024-04-01","2024-06-01"))
ds = geemap.ee_to_xarray(dataset.filterBounds(bbox)).squeeze().transpose("lat","lon","time").sel(lon=slice(*lonlim), lat=slice(*latlim)).load()
ds.to_netcdf(f"{path}/{dataset_name}_extended_2024.nc")

percentiles_NASA_GSFC_MERRA_slv_2_all_extended
percentiles_NASA_GSFC_MERRA_slv_2_apr_may_extended


In [104]:
path = f"../data/external/gpm_merra2"
dataset_name = dataset_id.replace("/","_")
dataset = ee.ImageCollection(dataset_id).select(variables).filter(ee.Filter.date("2024-04-01","2024-06-01"))
ds = geemap.ee_to_xarray(dataset.filterBounds(bbox)).squeeze().transpose("lat","lon","time").sel(lon=slice(*lonlim), lat=slice(*latlim)).load()
ds.to_netcdf(f"{path}/{dataset_name}_extended_2024.nc")

In [ ]:
percentiles = np.arange(0,100+1).astype("int").tolist()

datasets_all = {
    'NASA/GSFC/MERRA/slv/2': ["PS", "T10M", "TQV", "U10M", "V10M", "H500", "U500", "V500"],
    'NASA/GPM_L3/IMERG_V06': ["precipitationCal"],
}


In [13]:
lonlim = [-(88+39/60), -(30+40/60)]
latlim = [-(41+23/60), 12+58/60]
bbox = ee.Geometry.BBox(lonlim[0], latlim[0], lonlim[1], latlim[1])

In [155]:
for dataset_id in datasets_all:
    variables = datasets_all[dataset_id]
    dataset_name = dataset_id.replace("/","_")

    datasets = {
        f"percentiles_{dataset_name}_all": (
            ee.ImageCollection(dataset_id).select(variables)
                .reduce(ee.Reducer.percentile(percentiles))
        ),
        f"percentiles_{dataset_name}_apr_may": (
            ee.ImageCollection(dataset_id).select(variables)
                .filter(ee.Filter.dayOfYear(91, 151))
                .reduce(ee.Reducer.percentile(percentiles))
        ),
        f"mean_{dataset_name}_apr_may": (
            ee.ImageCollection(dataset_id).select(variables)
                .filter(ee.Filter.dayOfYear(91, 151)).mean()
        ),
    }

    if dataset_id=="NASA/GPM_L3/IMERG_V06":
        datasets = {**datasets, 
            f"percentiles_{dataset_name}_all_nonzero": (
                ee.ImageCollection(dataset_id).select(variables)
                    .map(lambda image: image.updateMask(image.gt(0)))
                    .reduce(ee.Reducer.percentile(percentiles))
            ),
            f"percentiles_{dataset_name}_apr_may_nonzero": (
                ee.ImageCollection(dataset_id).select(variables)
                    .filter(ee.Filter.dayOfYear(91, 151))
                    .map(lambda image: image.updateMask(image.gt(0)))
                    .reduce(ee.Reducer.percentile(percentiles))
            ),
        }
    
    for dataset in datasets:
        
        print(dataset)
        
        image = datasets[dataset]
        
        # Export the image, specifying the CRS, transform, and region.
        task = ee.batch.Export.image.toDrive(
            image=image,
            description=dataset,
            crs="EPSG:4326",
            folder="export",
            region=bbox,
        )
        task.start()

percentiles_NASA_GPM_L3_IMERG_V06_all_nonzero
percentiles_NASA_GPM_L3_IMERG_V06_apr_may_nonzero


Go to `https://code.earthengine.google.com/` and the `Tasks` tab. Run the other scripts after finishing the processes.